In [32]:
import pandas as pd
path =os.path.join('/data/yongchan/zero-shot-domain-specific-whisper/preprocessed_data/ACTER/en/htfl/annotated/annotations/sequential_annotations/iob_annotations/with_named_entities/htfl_en_001_seq_terms_nes.tsv')
lines = pd.read_csv(path,  sep='\t', skip_blank_lines=False, names=['word', 'boi'])
domain_terms = []
domain_term = []

for i in range(len(lines)):
    candidate_word = lines.iloc[i]['word']
    boi = lines.iloc[i]['boi']
    if boi == 'B':
        domain_term.append(candidate_word)
    elif boi == 'I':
        domain_term.append(candidate_word)
    else:
        if len(domain_term) > 0:
            domain_terms.append(domain_term)
            domain_term = []
            
if len(domain_term) > 0:
    domain_terms.append(domain_term)
    domain_term = []
            
        
print(domain_terms)

[['patient'], ['conditions'], ['patient'], ['Medicare', 'Patient', 'Safety', 'Monitoring', 'System'], ['MPSMS'], ['adverse', 'event'], ['patients', 'hospitalized'], ['acute', 'myocardial', 'infarction'], ['congestive', 'heart', 'failure'], ['pneumonia'], ['conditions'], ['surgery'], ['patients'], ['hospitals'], ['significant'], ['adverse', 'event'], ['acute', 'myocardial', 'infarction'], ['congestive', 'heart', 'failure'], ['in-hospital', 'adverse', 'events'], ['patients'], ['pneumonia'], ['surgical', 'conditions'], ['pressure', 'ulcers'], ['surgical', 'patients'], ['patient']]


# Create dataset with longer text

In [2]:
import os
import re
import numpy as np
import pandas as pd
from IPython.core.debugger import set_trace

train_dataset = pd.DataFrame(columns=['text', 'label', 'unique_label'])
master_path = '/data/yongchan/ATE/ACTER/en'

def normalize_texts(texts):
    text = ' '.join(texts)
    # Remove any empty space that are placed before special characters
    text = re.sub(r'\s+(?=[\W_])', '', text)
    
    # Remove any empty space that are placed after the character /
    text = re.sub(r'/(?=\s)', '/', text)
    return text

for dataset_name in os.listdir(master_path):
    rows = {}
    all_domain_terms = []
    all_texts = []
    all_term_types = []
    dataset_path = os.path.join(master_path, dataset_name, 'annotated')
    
    if dataset_name == 'cor':
        continue
    
    for category in os.listdir(dataset_path):
        category_path = os.path.join(dataset_path, category)
        if category == 'annotations':
            for seq_annotation in os.listdir(category_path):
                if seq_annotation == 'sequential_annotations':
                    iob_annotation_path = os.path.join(category_path, seq_annotation, 'iob_annotations', 'with_named_entities')
                    for filename in sorted(os.listdir(iob_annotation_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                        file_path = os.path.join(iob_annotation_path, filename)
                        lines = pd.read_csv(file_path,  sep='\t', names=['word', 'boi'])
                        domain_terms = []
                        domain_term = []
                        for i in range(len(lines)):
                            candidate_word = lines.iloc[i]['word']                                  
                            boi = lines.iloc[i]['boi']
                            
                            if candidate_word is np.nan and boi is not np.nan:
                                candidate_word = 'None'
                                
                            if boi == 'B':
                                if len(domain_term) == 0:
                                    domain_term.append(candidate_word)
                                else:
                                    domain_term = normalize_texts(domain_term)
                                    domain_terms.append(domain_term)
                                    domain_term = [candidate_word]
                            elif boi == 'I':
                                domain_term.append(candidate_word)
                            else:
                                if len(domain_term) > 0:
                                    domain_term = normalize_texts(domain_term)
                                    domain_terms.append(domain_term)
                                    domain_term = []
                                    
                        if len(domain_term) > 0:
                            domain_term = normalize_texts(domain_term)
                            domain_terms.append(domain_term)
                            domain_term = []
                        all_domain_terms.append(domain_terms)
                        
                elif seq_annotation == 'unique_annotation_lists':
                    unique_annotation_list_path = os.path.join(category_path, seq_annotation, f"{dataset_name}_en_terms_nes.tsv")
                    lines = pd.read_csv(unique_annotation_list_path,  sep='\t', names=['word', 'term_type'])
                    for i, domain_terms in enumerate(all_domain_terms):
                        term_types = []
                        for domain_term in domain_terms:
                            domain_term = domain_term.lower()
                            term_type = ''
                            count = 0
                            while len(term_type) == 0:                                    
                                if count==0:
                                    term_type = lines[lines['word']==domain_term]['term_type']
                                
                                elif count==1:
                                    character_removed_domain_term_wo_ws = re.sub(r'[^a-zA-Z0-9\s]', '', domain_term)
                                    term_type = lines[lines['word']==character_removed_domain_term_wo_ws]['term_type']
                                    
                                elif count==2:
                                    character_removed_domain_term_w_ws = re.sub(r'[^a-zA-Z0-9\s]', ' ', domain_term).split(' ')
                                    for i in sorted(range(len(character_removed_domain_term_w_ws)), reverse=True):
                                        for j in range(len(character_removed_domain_term_w_ws)):
                                            sliced_dash_removed_domain_term_w_ws = ' '.join(character_removed_domain_term_w_ws[j:j+i])
                                            term_type = lines[lines['word'].str.contains(sliced_dash_removed_domain_term_w_ws)]['term_type']
                                            if len(term_type) > 0:
                                                break
                                        if len(term_type) > 0:
                                            break
                                
                                elif count > 2:
                                    term_type = 'Not Annotated'                                            
                                    print(f'Path: {unique_annotation_list_path}, domain_term: {domain_term}, index: {i}')
                                    break
                                    
                                count += 1                                    

                            if not isinstance(term_type, str):
                                term_type = term_type.astype(str).values[0]
                            term_types.append(term_type)
                        all_term_types.append(term_types)
                                
        elif category == 'texts':
            for filename in sorted(os.listdir(category_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                with open(os.path.join(category_path, filename), 'r') as f:
                    text = f.read()
                    all_texts.append(text)
    
    rows = {'text': all_texts, 'label': all_domain_terms, 'unique_label': all_term_types}
    if dataset_name == 'htfl':
        rows['domain'] = ['Heart Failure']*len(all_texts)
        test_dataset = pd.DataFrame(rows)
    elif dataset_name == 'equi':
        rows['domain'] = ['Equitation']*len(all_texts)
        validation_dataset = pd.DataFrame(rows)
    else:
        dataset_name_to_domain = {'corp': 'Corruption', 'wind': 'Wind Energy'}
        rows['domain'] = [dataset_name_to_domain[dataset_name]]*len(all_texts)
        train_dataset = pd.concat([train_dataset, pd.DataFrame(rows)])
        

In [3]:
print(f'Train dataset length: {len(train_dataset)}')
print(f'Validation dataset length: {len(validation_dataset)}')
print(f'Test dataset length: {len(test_dataset)}')

Train dataset length: 17
Validation dataset length: 34
Test dataset length: 190


In [16]:
# pd.set_option('display.max_colwidth', 500)

# test_dataset.head(10)

In [5]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_dataset, preserve_index=False)
test_dataset = Dataset.from_pandas(test_dataset, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_dataset, preserve_index=False)

In [123]:
from datasets import Dataset, DatasetDict

dataset_save_path = '/data/yongchan/ATE/ACTER/huggingface/long'
os.makedirs(dataset_save_path, exist_ok=True)

dataset = DatasetDict({'train': train_dataset, 'validation': validation_dataset, 'test': test_dataset})
dataset.save_to_disk(dataset_save_path)

Saving the dataset (0/1 shards):   0%|          | 0/17 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/34 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/190 [00:00<?, ? examples/s]

# Create dataset with shorter text

In [11]:
import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.core.debugger import set_trace

train_dataset = pd.DataFrame(columns=['text', 'label', 'unique_label'])
master_path = '/data/yongchan/ATE/ACTER/en'

def normalize_texts(texts):
    text = ' '.join(texts)
    # Remove any empty space that are placed before special characters
    text = re.sub(r'\s+(?=[\W_])', '', text)
    
    # Remove any empty space that are placed after the character /
    text = re.sub(r'/(?=\s)', '/', text)
    return text

def remove_special_characters(text):
    text = text.lower().replace(' ', '')
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

for dataset_name in os.listdir(master_path):
    rows = {}
    all_domain_terms = []
    all_candidate_words = []
    all_texts = {}
    all_term_types = []
    dataset_path = os.path.join(master_path, dataset_name, 'annotated')
    
    if dataset_name == 'cor':
        continue
    
    for category in sorted(os.listdir(dataset_path)):
        category_path = os.path.join(dataset_path, category)
        
        if category == 'annotations':
            for seq_annotation in sorted(os.listdir(category_path)):                                
                if seq_annotation == 'sequential_annotations':
                    iob_annotation_path = os.path.join(category_path, seq_annotation, 'iob_annotations', 'with_named_entities')
                    for filename in sorted(os.listdir(iob_annotation_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                        file_path = os.path.join(iob_annotation_path, filename)
                        lines = pd.read_csv(file_path,  sep='\t', na_values=[], skip_blank_lines=False, names=['word', 'boi'])
                    
                        split_indexes = lines[pd.isna(lines[['word', 'boi']]).all(axis=1)].index
                        for prev_index, split_index in zip([-1]+list(split_indexes[:-1]), split_indexes):
                            split_lines = lines.iloc[prev_index+1:split_index]
                            domain_terms = []
                            domain_term = []
                            candidate_words = []
                            for i in range(len(split_lines)):
                                candidate_word = split_lines.iloc[i]['word']                                    
                                boi = split_lines.iloc[i]['boi']
                                if candidate_word is np.nan and boi is not np.nan:
                                    candidate_word = 'None'
                                
                                if boi == 'B':
                                    if len(domain_term) == 0:
                                        domain_term.append(candidate_word)
                                    else:
                                        domain_term = normalize_texts(domain_term)
                                        domain_terms.append(domain_term)
                                        domain_term = [candidate_word]
                                elif boi == 'I':
                                    domain_term.append(candidate_word)
                                else:
                                    if len(domain_term) > 0:
                                        domain_term = normalize_texts(domain_term)
                                        domain_terms.append(domain_term)
                                        domain_term = []
                                candidate_words.append(candidate_word.lower())
                            
                            if len(domain_term) > 0:
                                domain_term = normalize_texts(domain_term)
                                domain_terms.append(domain_term)
                                                                    
                            all_domain_terms.append(domain_terms)
                            # all_candidate_words.append(candidate_words)
                            all_candidate_words.append(remove_special_characters(''.join(candidate_words)))
                            
                elif seq_annotation == 'unique_annotation_lists':
                    unique_annotation_list_path = os.path.join(category_path, seq_annotation, f"{dataset_name}_en_terms_nes.tsv")
                    lines = pd.read_csv(unique_annotation_list_path,  sep='\t', names=['word', 'term_type'])
                    for i, domain_terms in enumerate(all_domain_terms):
                        term_types = []
                        for domain_term in domain_terms:
                            domain_term = domain_term.lower()
                            term_type = ''
                            count = 0
                            while len(term_type) == 0:                                    
                                if count==0:
                                    term_type = lines[lines['word']==domain_term]['term_type']
                                
                                elif count==1:
                                    character_removed_domain_term_wo_ws = re.sub(r'[^a-zA-Z0-9\s]', '', domain_term)
                                    term_type = lines[lines['word']==character_removed_domain_term_wo_ws]['term_type']
                                    
                                elif count==2:
                                    character_removed_domain_term_w_ws = re.sub(r'[^a-zA-Z0-9\s]', ' ', domain_term).split(' ')
                                    for i in sorted(range(len(character_removed_domain_term_w_ws)), reverse=True):
                                        for j in range(len(character_removed_domain_term_w_ws)):
                                            sliced_dash_removed_domain_term_w_ws = ' '.join(character_removed_domain_term_w_ws[j:j+i])
                                            term_type = lines[lines['word'].str.contains(sliced_dash_removed_domain_term_w_ws)]['term_type']
                                            if len(term_type) > 0:
                                                break
                                        if len(term_type) > 0:
                                            break
                                
                                elif count > 2:
                                    term_type = 'Not Annotated'                                            
                                    print(f'Path: {unique_annotation_list_path}, domain_term: {domain_term}, index: {i}')
                                    break
                                    
                                count += 1                                    

                            if not isinstance(term_type, str):
                                term_type = term_type.astype(str).values[0]
                            term_types.append(term_type)
                        all_term_types.append(term_types)
        
        # TODO: 여기 고치기...
        elif category == 'texts':
            i =0
            for filename in sorted(os.listdir(category_path), key=lambda x: int(os.path.splitext(x)[0].split('_')[2])):
                print(f'###### Filename: {filename} is processing... #######')
                with open(os.path.join(category_path, filename), 'r') as f:
                    text = f.read()
                    texts = text.split('\n')                            
                    texts_copy = texts.copy()
                    for text_idx in range(len(texts)):
                        while '\n' in texts[text_idx]:
                            texts[text_idx] = texts[text_idx].replace('\n', '')
                    
                    while '' in texts:
                        texts.remove('')
                        
                    modified_texts = list(map(remove_special_characters, texts))
                    modified_texts_to_texts={modified_text: text for modified_text, text in zip(modified_texts, texts)}
                    temp_all_texts = []
                    temp_all_domain_terms = []
                    temp_all_term_types = []
                    j=0
                    while i < len(all_candidate_words):
                        all_candidate_word = all_candidate_words[i]
                        # if j==286 and filename == 'wind_en_02.txt':
                        #     print('idk')
                        for idx, modified_text in enumerate(modified_texts[j:]):
                            if all_candidate_word in modified_text:
                                temp_all_texts.append(modified_text)
                                domain_term = all_domain_terms[i]
                                temp_all_domain_terms.append(domain_term)
                                term_types = all_term_types[i]
                                temp_all_term_types.append(term_types)
                                if all_candidate_word == modified_text:
                                    j+=1
                                elif i < len(all_candidate_words)-1 and all_candidate_words[i+1] not in modified_text:
                                    j+=1
                                break
                            elif len(modified_text[j:]) <= 1:
                                j+=1
                        
                        i+=1
                        if len(modified_texts[j:]) == 0 or idx == len(modified_texts[j:]) and modified_texts[j:][-1] == all_candidate_word:
                            for modified_text, domain_terms, term_types in zip(temp_all_texts, temp_all_domain_terms, temp_all_term_types):
                                text = modified_texts_to_texts[modified_text]
                                if text not in all_texts:
                                    all_texts[text] = [[],[]]
                                all_texts[text][0].extend(domain_terms)
                                all_texts[text][1].extend(term_types)
                            for text in texts:
                                if text not in list(all_texts.keys()):
                                    print(f'Text: {text} is not in all_texts')
                            print(f'###### Filename: {filename} is finished! #######\n')
                            break
                        
            # if len_texts != len(all_texts):
            #     raise ValueError(f'Length of texts: {len_texts} is not equal to length of all_texts: {len(all_texts)}')
    
    all_text_values = all_texts.values()
    all_domain_terms, all_term_types = [], []
    for all_text_value in all_text_values:
        all_domain_terms.append(all_text_value[0])
        all_term_types.append(all_text_value[1])
        
    rows = {'text': list(all_texts.keys()), 'label': all_domain_terms, 'unique_label': all_term_types}
    rows['dataset_name'] = [dataset_name]*len(all_texts)
    if dataset_name == 'htfl':
        rows['domain'] = ['Heart Failure']*len(all_texts)
        test_dataset = pd.DataFrame(rows)
    elif dataset_name == 'equi':
        rows['domain'] = ['Equitation']*len(all_texts)
        validation_dataset = pd.DataFrame(rows)
    else:
        dataset_name_to_domain = {'corp': 'Corruption', 'wind': 'Wind Energy'}
        rows['domain'] = [dataset_name_to_domain[dataset_name]]*len(all_texts)
        dataset_name_to_domain = {'corp': 'Corruption', 'wind': 'Wind Energy'}
        train_dataset = pd.concat([train_dataset, pd.DataFrame(rows)])
        

###### Filename: wind_en_01.txt is processing... #######
Text: : is not in all_texts
Text: x is not in all_texts
Text: λ = 10 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . is not in all_texts
Text: CP is not in all_texts
Text: P is not in all_texts
Text: w′ is not in all_texts
Text: T is not in all_texts
Text: B is not in all_texts
Text: a is not in all_texts
Text: λ is not in all_texts
Text: λd λr is not in all_texts
Text: ci is not in all_texts
Text: ρ is not in all_texts
Text: Ω is not in all_texts
Text: α is not in all_texts
Text: θ is not in all_texts
Text: θi is not in all_texts
Text: ϕ is not in all_texts
Text: σ ν is not in all_texts
Text: γ Γ is not in all_texts
Text: δΓ is not in all_texts
Text: Re is not in all_texts
Text: 6 is not in all_texts
Text: 12 is not in all_texts
Text: (a) is not in all_texts
Text: (a) is not in all_texts
Text: λ= is not in all_texts
Text: CT = is not in all_texts
Text: a= is not in all_texts
Text: P=

In [13]:
print(f'Train dataset length: {len(train_dataset)}')
print(f'Validation dataset length: {len(validation_dataset)}')
print(f'Test dataset length: {len(test_dataset)}')

Train dataset length: 2326
Validation dataset length: 586
Test dataset length: 754


In [17]:
# pd.set_option('display.max_colwidth', 500)
# test_dataset.tail(10)

In [18]:
# train_dataset.tail(10)

In [128]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_dataset, preserve_index=False)
test_dataset = Dataset.from_pandas(test_dataset, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_dataset, preserve_index=False)

In [129]:
from datasets import Dataset, DatasetDict

dataset_save_path = '/data/yongchan/ATE/ACTER/huggingface/short'
os.makedirs(dataset_save_path, exist_ok=True)

dataset = DatasetDict({'train': train_dataset, 'validation': validation_dataset, 'test': test_dataset})
dataset.save_to_disk(dataset_save_path)

Saving the dataset (0/1 shards):   0%|          | 0/2326 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/586 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/754 [00:00<?, ? examples/s]